<a href="https://colab.research.google.com/github/Mitochondrion3000/epic-competiton/blob/claude/gracious-brown-4ybsmj/simple_model/step4_test_and_submission.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Простая интерпретируемая модель. Шаг 4: тест, официальные метрики и сабмит

Берём финальные модели шага 3б и предсказываем **все позиции тестового шаблона** (у *Nematostella* это 72.6 млн
позиций-цепей на трёх хромосомах), в том порядке, который ждёт скорер. Для учебного вида есть ответы теста, поэтому
считаем официальные метрики (на двух половинах теста, как на лидерборде) и сравниваем с CNN.
В конце пишем файл сабмита и смотрим на пару участков генома глазами.

Для своего вида (`Ofas_2.0`) последовательность та же: шаги 1 → 2 → 3 → 3б → 4 с `EPIC_SPECIES=Ofas_2.0`;
ответов теста там нет, и раздел с оценкой просто пропустится.

## 0. Настройки, модели, шаблон теста

In [ ]:
import os, sys, gzip, json, hashlib, tarfile, time, urllib.request
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
try:
    import lightgbm as lgb
except ImportError:
    os.system("pip install -q lightgbm"); import lightgbm as lgb

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

SPECIES = os.environ.get("EPIC_SPECIES", "jaNemVect1.1")
ON_COLAB = "google.colab" in sys.modules
if ON_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
DATA_DIR = os.environ.get("EPIC_DATA_DIR", "/content/epic")
OUT_DIR = os.environ.get("EPIC_OUT_DIR", "/content/drive/MyDrive/epic_simple" if ON_COLAB else "/content/epic_simple")
BASE, FP = f"{DATA_DIR}/{SPECIES}", f"{DATA_DIR}/{SPECIES}/for_participants"

RAW = "https://raw.githubusercontent.com/Mitochondrion3000/epic-competiton/claude/gracious-brown-4ybsmj/simple_model/"
for mod in ("features.py", "motifs.py", "scoring.py"):
    if not os.path.exists(mod): urllib.request.urlretrieve(RAW + mod, mod)
import features as F, motifs as M, scoring as S

P = json.load(open(f"{OUT_DIR}/{SPECIES}.step3b_params.json"))
FEATURES = P["features"]
PARAMS = {"pwm": np.array(P["pwm"]), "kmers": [tuple(k) for k in P["kmers"]], "tf": P["tf"], "wide": P["wide"], "local": P["local"]}
SUMMARY = json.load(open(f"{OUT_DIR}/{SPECIES}.step3b_summary.json"))
ALPHA = SUMMARY["alpha"]
MODELS = {tag: (lgb.Booster(model_file=f"{OUT_DIR}/{SPECIES}.step3b_{tag}_cls.txt"),
                lgb.Booster(model_file=f"{OUT_DIR}/{SPECIES}.step3b_{tag}_reg.txt")) for tag in ("additive", "depth3")}
print(f"{len(FEATURES)} признаков, модели: {list(MODELS)}, alpha = {ALPHA}")

In [ ]:
def download_dataset(species, data_dir):
    if os.path.isdir(f"{data_dir}/{species}"): return
    import requests
    os.makedirs(data_dir, exist_ok=True)
    rec = requests.get("https://zenodo.org/api/records/22285753", timeout=60).json()
    f = [x for x in rec["files"] if x["key"] == f"{species}.tgz"][0]
    dst, md5 = f"{data_dir}/{species}.tgz", hashlib.md5()
    with requests.get(f["links"]["self"], stream=True, timeout=120) as r, open(dst, "wb") as fh:
        r.raise_for_status()
        for chunk in tqdm(r.iter_content(1 << 22), total=f["size"] // (1 << 22) + 1, desc=f"скачиваю {species}"):
            fh.write(chunk); md5.update(chunk)
    assert md5.hexdigest() == f.get("checksum", "md5:").split(":")[-1], "MD5 не совпал"
    with tarfile.open(dst) as tf:
        try: tf.extractall(data_dir, filter="data")
        except TypeError: tf.extractall(data_dir)
    os.remove(dst)

CODE = np.full(256, 4, np.uint8)
for i, b in enumerate("ACGT"):
    CODE[ord(b)] = CODE[ord(b.lower())] = i

def read_genome(path, keep):
    seq, low, name, chunks = {}, {}, None, []
    def flush():
        if name in keep:
            raw = np.frombuffer(b"".join(chunks), np.uint8)
            seq[name], low[name] = CODE[raw], raw >= ord("a")
    with gzip.open(path, "rb") as fh:
        for line in fh:
            if line.startswith(b">"):
                flush(); name, chunks = line[1:].split()[0].decode(), []
            elif name in keep:
                chunks.append(line.rstrip())
    flush()
    return seq, low

download_dataset(SPECIES, DATA_DIR)
tpl = pd.read_csv(f"{FP}/template.test.bed.gz", sep="\t", header=None, usecols=[0, 1, 2, 5],
                  names=["chrom", "start", "end", "strand"], dtype={"chrom": str, "strand": str})
tpl["len"] = tpl.end - tpl.start
tpl["off"] = np.r_[0, np.cumsum(tpl.len.values)[:-1]]       # где в итоговом векторе начинается каждая строка шаблона
N_TOTAL = int(tpl.len.sum())
SEQ, LOW = read_genome(f"{BASE}/genome/genome.fa.gz", set(tpl.chrom))
print(f"шаблон: {len(tpl):,} строк, {N_TOTAL:,} позиций-цепей, контиги: {list(dict.fromkeys(tpl.chrom))}")
print("порядок цепей в шаблоне:", tpl.strand.iloc[[0, len(tpl) // 2 - 1, len(tpl) // 2, -1]].tolist(), "(сначала все '+', потом все '−')")

## 1. Предсказание всех позиций теста

Для каждого контига и каждой цепи собираем позиции всех строк шаблона, считаем 61 признак той же функцией, что на
обучении (`features.feature_matrix`), прогоняем обе модели и раскладываем счёт в порядок шаблона.
Счёт модели = `log P(старт) + E[log(1 + счёт) | старт]`. Позиции с `N` в геноме получают самый низкий счёт.
Занимает около 15–20 минут, пик памяти ~4 ГБ.

In [ ]:
SCORES = {tag: np.zeros(N_TOTAL, np.float32) for tag in MODELS}
NMASK = np.zeros(N_TOTAL, bool)
t0 = time.time()
for (c, st), g in tqdm(tpl.groupby(["chrom", "strand"], sort=False), desc="контиг × цепь"):
    pos = np.concatenate([np.arange(s, e) for s, e in zip(g.start.values, g.end.values)])
    dst = np.concatenate([np.arange(o, o + n) for o, n in zip(g.off.values, g.len.values)])
    X = F.feature_matrix(SEQ[c], LOW[c], pos, np.full(len(pos), st), PARAMS, FEATURES)
    for tag, (clf, reg) in MODELS.items():
        SCORES[tag][dst] = np.log(np.clip(clf.predict(X), 1e-12, 1)) + reg.predict(X)
    NMASK[dst] = SEQ[c][pos] == 4
    del X
    print(f"{c} {st}: {len(pos):,} позиций, {(time.time() - t0) / 60:.1f} мин", flush=True)
for tag, s in SCORES.items():
    s[NMASK] = -np.inf
    np.save(f"{OUT_DIR}/{SPECIES}.step4_{tag}_score.npy", s)
print(f"готово: {N_TOTAL:,} позиций, N-позиций {NMASK.sum():,}")

## 2. Официальные метрики на тесте

Скорер оценивает каждую половину теста отдельно (`mask1` и `mask2`, это половины, как на публичном и приватном
лидерборде), итог — среднее по половинам. Перед оценкой счёт квантуется так же, как будет записан в файл
(`rank_power` с `alpha` из шага 3б).

In [ ]:
HAS_GT = os.path.exists(f"{BASE}/ground_truth/csRNA-test.txt.gz")
if HAS_GT:
    read_col = lambda p, dt: pd.read_csv(p, header=None, dtype=dt)[0].values
    GT = read_col(f"{BASE}/ground_truth/csRNA-test.txt.gz", np.float32)
    M1 = read_col(f"{BASE}/scoring/mask1-test.txt.gz", np.int8).astype(bool); M2 = read_col(f"{BASE}/scoring/mask2-test.txt.gz", np.int8).astype(bool)
    assert len(GT) == N_TOTAL
    rows = {}
    for tag, name in (("additive", "аддитивная модель"), ("depth3", "деревья глубины 3")):
        sub = S.rank_power(SCORES[tag], ALPHA, NMASK)
        r = {h: S.official_score(sub, GT, m) for h, m in (("half1", M1), ("half2", M2))}
        rows[f"НАША: {name}"] = {"AUPRC": np.mean([r[h]["AUPRC"] for h in r]), "Spearman": np.mean([r[h]["Spearman"] for h in r]),
                                 "AUPRC half1": r["half1"]["AUPRC"], "AUPRC half2": r["half2"]["AUPRC"],
                                 "Spearman half1": r["half1"]["Spearman"], "Spearman half2": r["half2"]["Spearman"]}
    REF = {"случайный уровень": (float((GT > 0).mean()), 0.0), "динуклеотиды (бейзлайн организаторов)": (0.00116, 0.0361),
           "PWM старта (EDA, без квантования)": (0.0019, 0.153), "dilated CNN (наш прошлый пайплайн)": (0.0434, 0.379),
           "InitiationNet big 96k (лучшая CNN)": (0.13165, 0.391)}
    cmp = pd.concat([pd.DataFrame({k: {"AUPRC": a, "Spearman": s} for k, (a, s) in REF.items()}).T, pd.DataFrame(rows).T])
    display(cmp.style.format("{:.4f}", na_rep="—"))
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.4))
    for a, met in zip(ax, ("AUPRC", "Spearman")):
        v = cmp[met]; cols = ["tab:red" if i.startswith("НАША") else "tab:gray" for i in cmp.index]
        a.barh(cmp.index, v, color=cols); a.set_title(f"{met}: тест {SPECIES}")
        for i, x in enumerate(v): a.text(x, i, f" {x:.3f}", va="center", fontsize=8)
        if met == "AUPRC": a.set_xscale("log")
    plt.tight_layout(); plt.show()
else:
    print("ответов теста нет (это не учебный вид) — оценку пропускаем")

COMMENT_TEST

## 3. Как это выглядит на геноме

Три участка тестовой хромосомы вокруг сильных стартов: сверху настоящие старты (счёт csRNA-seq на каждой цепи), ниже
счёт модели, ещё ниже — сайты факторов и состав, из которых модель этот счёт собрала. Так видно, *почему* модель
поставила высокий счёт в одном месте и низкий в соседнем.

In [ ]:
if HAS_GT:
    tag = "depth3"
    # находим строки шаблона, где лежат самые сильные старты, и рисуем окно ±400 нт вокруг
    top_idx = np.argsort(-GT)[:2000]
    picked, used = [], []
    for k in top_idx:
        r = np.searchsorted(tpl.off.values, k, side="right") - 1
        c, st, pos = tpl.chrom.iat[r], tpl.strand.iat[r], tpl.start.iat[r] + (k - tpl.off.iat[r])
        if all(c != u[0] or abs(pos - u[1]) > 50_000 for u in used):
            used.append((c, pos)); picked.append((c, pos))
        if len(picked) == 3: break

    def track(c, lo, hi, vec):
        out = np.full((2, hi - lo), np.nan, np.float32)
        g = tpl[(tpl.chrom == c) & (tpl.end > lo) & (tpl.start < hi)]
        for s, e, st, o in zip(g.start.values, g.end.values, g.strand.values, g.off.values):
            a, b = max(s, lo), min(e, hi)
            out[0 if st == "+" else 1, a - lo:b - lo] = vec[o + a - s:o + b - s]
        return out

    fig, axs = plt.subplots(3 * len(picked), 1, figsize=(15, 3.2 * len(picked)), sharex=False,
                            gridspec_kw={"height_ratios": [1.2, 1.2, 1.0] * len(picked)})
    for n, (c, center) in enumerate(picked):
        lo, hi = center - 400, center + 400; x = np.arange(lo, hi)
        gt, sc = track(c, lo, hi, GT), track(c, lo, hi, SCORES[tag])
        a0, a1, a2 = axs[3 * n], axs[3 * n + 1], axs[3 * n + 2]
        a0.bar(x, np.nan_to_num(gt[0]), color="tab:red", width=1, label="старты на +"); a0.bar(x, -np.nan_to_num(gt[1]), color="tab:blue", width=1, label="старты на −")
        a0.set_yscale("symlog"); a0.set_title(f"{c}:{lo:,}-{hi:,}", fontsize=9); a0.legend(fontsize=7, loc="upper right"); a0.set_ylabel("счёт")
        a1.plot(x, sc[0], color="tab:red", lw=.8, label="модель, +"); a1.plot(x, sc[1], color="tab:blue", lw=.8, label="модель, −")
        a1.set_ylabel("счёт модели"); a1.legend(fontsize=7, loc="upper right")
        s = SEQ[c][lo:hi]; gcw = np.convolve(((s == 1) | (s == 2)).astype(float), np.ones(51) / 51, mode="same")
        a2.fill_between(x, 0, gcw, color="lightgray", label="GC (окно 51)"); a2.set_ylim(0, 1.05)
        a2.fill_between(x, 0, LOW[c][lo:hi] * 0.15, color="gray", label="повтор")
        for k, m in enumerate(P["tf"][:6]):
            # сайт фактора двуцепочечный (счёт уже берёт максимум по двум ориентациям), поэтому ищем по + цепи
            h = np.flatnonzero(M.score_track(SEQ[c][lo:hi], np.array(m["pwm"], np.float32)) >= m["thr"])
            a2.scatter(lo + h, np.full(len(h), 0.3 + 0.11 * k), marker="|", s=80, color=plt.cm.tab10(k), label=m["name"])
        a2.legend(fontsize=6, ncol=8, loc="upper left"); a2.set_yticks([])
    plt.tight_layout(); plt.show()

COMMENT_BROWSER

## 4. Файл сабмита

Лучшая модель (деревья глубины 3), квантование `rank_power(alpha)`, 5 знаков после запятой, одна строка на позицию
в порядке шаблона, `N` → 0. Перед загрузкой файл стоит проверить официальным валидатором:
https://github.com/autosome-ru/EPIC_scoring

In [ ]:
SUB = S.rank_power(SCORES["depth3"], ALPHA, NMASK)
sub_path = f"{OUT_DIR}/{SPECIES}.simple_model.predictions.txt.gz"
S.write_submission(SUB, sub_path)
with gzip.open(sub_path, "rt") as fh:
    head = [next(fh).strip() for _ in range(3)]
checks = {"строк = позиций шаблона": len(SUB) == N_TOTAL, "все значения в [0, 1]": bool((SUB >= 0).all() and (SUB <= 1).all()),
          "N-позиции = 0": bool((SUB[NMASK] == 0).all()), "первые строки": head,
          "различных значений": int(len(np.unique(S.quantise(SUB)))), "alpha": ALPHA,
          "размер файла, МБ": round(os.path.getsize(sub_path) / 1e6, 1)}
display(pd.Series(checks, name="проверка").to_frame())
assert checks["строк = позиций шаблона"] and checks["все значения в [0, 1]"]
if HAS_GT:
    print("контроль: метрики по записанному вектору", {h: {k: round(v, 4) for k, v in S.official_score(SUB, GT, m).items()} for h, m in (("half1", M1), ("half2", M2))})
print("сабмит:", sub_path)

## Итог серии

COMMENT_SUMMARY